# Unit 2 Assignment: Mixture of Experts (MoE) Router

This notebook builds a **Smart Customer Support Router** using a Mixture of Experts pattern with Groq.

Implemented:
- Expert configurations (`technical`, `billing`, `sales`, `general`)
- LLM-powered intent router (`route_prompt`)
- Orchestrator (`process_request`)
- Bonus `tool` route for mock Bitcoin price lookup

In [14]:
%pip install -q groq python-dotenv

import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

try:
    from groq import Groq
except Exception as e:
    raise RuntimeError(f"Could not import groq package: {e}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
ENABLE_LLM = bool(GROQ_API_KEY)
client = Groq(api_key=GROQ_API_KEY) if ENABLE_LLM else None

if ENABLE_LLM:
    print("Groq client ready.")
else:
    print("No GROQ_API_KEY found. Notebook will run in fallback mode for demonstration.")

Note: you may need to restart the kernel to use updated packages.
Groq client ready.


## What This Setup Cell Does

- Installs required libraries for Groq and environment loading.
- Loads values from the local .env file.
- Builds a Groq client if GROQ_API_KEY is available.
- Enables fallback mode if no key is available so the notebook still runs.

In [15]:
BASE_MODEL = "mixtral-8x7b-32768"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": (
            "You are a Technical Support Expert. Be precise, code-focused, and diagnostic. "
            "Provide reproducible troubleshooting steps and minimal, correct fixes."
        ),
        "temperature": 0.7,
    },
    "billing": {
        "system_prompt": (
            "You are a Billing Support Expert. Be empathetic, policy-driven, and clear about "
            "charges, refunds, timelines, and next actions."
        ),
        "temperature": 0.7,
    },
    "sales": {
        "system_prompt": (
            "You are a Sales Expert. Be concise, discovery-oriented, and value-focused. "
            "Recommend best-fit plans/features and ask one relevant qualifying question."
        ),
        "temperature": 0.7,
    },
    "general": {
        "system_prompt": (
            "You are a General Support Expert for casual or uncategorized requests. "
            "Be friendly and helpful."
        ),
        "temperature": 0.7,
    },
}

VALID_CATEGORIES = ["technical", "billing", "sales", "general", "tool"]
print("Configured experts:", ", ".join(MODEL_CONFIG.keys()))

Configured experts: technical, billing, sales, general


## What This Configuration Cell Does

- Defines the base model.
- Creates specialist expert profiles using different system prompts.
- Declares valid routing categories used by the router and orchestrator.

In [ ]:
from datetime import datetime, timezone


def call_groq(messages, temperature=0.7, max_tokens=512):
    """Thin wrapper around Groq chat completions."""
    if not ENABLE_LLM:
        raise RuntimeError("Groq API unavailable (missing GROQ_API_KEY).")

    completion = client.chat.completions.create(
        model=BASE_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message.content.strip()


def heuristic_route(user_input):
    """Fallback router for offline/demo execution."""
    text = user_input.lower()

    if any(k in text for k in ["bitcoin", "btc", "price", "current price"]):
        return "tool"
    if any(k in text for k in ["error", "bug", "exception", "stack trace", "python", "code"]):
        return "technical"
    if any(k in text for k in ["charged", "refund", "invoice", "billing", "subscription", "payment"]):
        return "billing"
    if any(k in text for k in ["pricing", "plan", "demo", "buy", "quote", "enterprise"]):
        return "sales"
    return "general"


def route_prompt(user_input):
    """
    LLM router that returns ONLY one category from:
    technical, billing, sales, general, tool
    """
    router_prompt = (
        "Classify the user input into exactly one category from this list: "
        "[technical, billing, sales, general, tool].\n"
        "Definitions:\n"
        "- technical: bugs, errors, code, debugging\n"
        "- billing: charges, invoices, refunds, payments\n"
        "- sales: new plans, pricing, demos, purchase intent\n"
        "- tool: asks for live/current bitcoin price\n"
        "- general: casual chat or anything else\n\n"
        "Return ONLY the category name, nothing else."
    )

    try:
        raw = call_groq(
            [
                {"role": "system", "content": router_prompt},
                {"role": "user", "content": user_input},
            ],
            temperature=0,
            max_tokens=8,
        ).lower().strip()

        category = raw.split()[0].strip(".,:;!?[](){}\"'")
        return category if category in VALID_CATEGORIES else "general"
    except Exception:
        return heuristic_route(user_input)


def fetch_bitcoin_price_mock():
    """Bonus tool: mock external data source."""
    return {
        "asset": "BTC",
        "currency": "USD",
        "price": 67234.12,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "source": "mock-feed",
    }


def run_expert(category, user_input):
    if category == "tool":
        data = fetch_bitcoin_price_mock()
        return (
            f"Current {data['asset']} price: ${data['price']:,.2f} {data['currency']} "
            f"(source: {data['source']}, at {data['timestamp']})."
        )

    config = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])

    try:
        return call_groq(
            [
                {"role": "system", "content": config["system_prompt"]},
                {"role": "user", "content": user_input},
            ],
            temperature=config["temperature"],
            max_tokens=512,
        )
    except Exception:
        # Deterministic fallback if API is unavailable
        if category == "technical":
            return "[Technical Expert Fallback] Please share traceback, runtime version, and minimal reproducible code. Start by isolating failing line and validating inputs."
        if category == "billing":
            return "[Billing Expert Fallback] I can help with billing/refund flow. Please provide invoice ID, charge date, and last 4 digits of payment method."
        if category == "sales":
            return "[Sales Expert Fallback] Happy to help with plans and pricing. What team size and primary use-case do you have?"
        return "[General Expert Fallback] I can help. Could you share a bit more context?"


def process_request(user_input):
    """Main orchestrator for MoE routing and expert response."""
    category = route_prompt(user_input)
    answer = run_expert(category, user_input)
    return {
        "category": category,
        "answer": answer,
    }

## What The Core Logic Cell Does

- call_groq: thin wrapper for Groq chat completions.
- route_prompt: classifies user intent into one category.
- run_expert: dispatches to the selected expert persona.
- process_request: orchestrates route + expert response.
- Includes a bonus tool route for Bitcoin price queries.

In [17]:
samples = [
    "My python script throws IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Can you suggest the best plan for a 20-person startup?",
    "Hey, what can you do?",
    "What is the current price of Bitcoin?",
]

for q in samples:
    result = process_request(q)
    print("=" * 80)
    print(f"User: {q}")
    print(f"Routed To: {result['category']}")
    print(f"Answer: {result['answer']}")

User: My python script throws IndexError on line 5.
Routed To: technical
Answer: [Technical Expert Fallback] Please share traceback, runtime version, and minimal reproducible code. Start by isolating failing line and validating inputs.
User: I was charged twice for my subscription this month.
Routed To: billing
Answer: [Billing Expert Fallback] I can help with billing/refund flow. Please provide invoice ID, charge date, and last 4 digits of payment method.
User: Can you suggest the best plan for a 20-person startup?
Routed To: sales
Answer: [Sales Expert Fallback] Happy to help with plans and pricing. What team size and primary use-case do you have?
User: Hey, what can you do?
Routed To: general
Answer: [General Expert Fallback] I can help. Could you share a bit more context?
User: What is the current price of Bitcoin?
Routed To: tool
Answer: Current BTC price: $67,234.12 USD (source: mock-feed, at 2026-03-28T13:57:04.778217+00:00).


In [18]:
# Quick routing sanity checks
assert route_prompt("IndexError in my Python code") in VALID_CATEGORIES
assert route_prompt("Need refund for accidental charge") in VALID_CATEGORIES
assert process_request("current price of bitcoin")["category"] in VALID_CATEGORIES
print("Sanity checks passed.")

Sanity checks passed.


## Notes

- Router uses `temperature=0` for consistency.
- Experts use `temperature=0.7` for richer responses.
- Bonus `tool` expert handles Bitcoin price queries via a mock data function.
- To use live Groq responses, set `GROQ_API_KEY` in `.env`.